# E-commerce Sales & Customer Analysis

**Junior Data Analyst portfolio project** — Python, Pandas, NumPy and Matplotlib.

The dataset is synthetic and includes a few intentional data-quality issues for validation practice.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / 'src'))
from analysis_utils import total_revenue, average_order_value, repeat_customer_rate


## 1. Load and inspect

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'ecommerce_orders_raw.csv', parse_dates=['order_date'])
display(df.head())
print('Shape:', df.shape)
df.info()


## 2. Data-quality checks

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
display(df.isna().sum().sort_values(ascending=False))
print('Negative revenue:', (df['revenue_eur'] < 0).sum())
print('Invalid quantity:', (~df['quantity'].between(1, 20)).sum())


## 3. Clean and validate

In [ ]:
clean = df.drop_duplicates().copy()
clean['region'] = clean['region'].fillna('Unknown')
clean['payment_method'] = clean['payment_method'].fillna('Unknown')
clean['discount_pct'] = clean['discount_pct'].fillna(0)
clean['revenue_eur'] = (clean['unit_price'] * clean['quantity'] * (1 - clean['discount_pct'])).round(2)

assert clean['order_id'].is_unique
assert clean['revenue_eur'].ge(0).all()
print('Rows after cleaning:', len(clean))


## 4. KPI analysis

In [ ]:
completed = clean[clean['order_status'].eq('Completed')].copy()
kpis = pd.Series({
    'Completed revenue (€)': total_revenue(clean),
    'Completed orders': completed['order_id'].nunique(),
    'Units sold': completed['quantity'].sum(),
    'Average order value (€)': average_order_value(clean),
    'Repeat customer rate (%)': repeat_customer_rate(clean),
    'Cancellation rate (%)': clean['order_status'].eq('Cancelled').mean() * 100,
    'Return rate (%)': clean['order_status'].eq('Returned').mean() * 100,
})
display(kpis.round(2).to_frame('value'))


## 5. Revenue by category and region

In [ ]:
category = completed.groupby('category')['revenue_eur'].sum().sort_values(ascending=False)
region = completed.groupby('region')['revenue_eur'].sum().sort_values(ascending=False)
display(category.to_frame('revenue_eur'))
display(region.to_frame('revenue_eur'))


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(category.index, category.values)
plt.title('Revenue by Category')
plt.ylabel('Revenue (€)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 6. Business conclusions

- Separate completed revenue from cancelled/returned orders to avoid overstating performance.
- Compare category, region and channel KPIs to identify high-value segments.
- Keep data-quality checks as a repeatable pre-reporting step.
- Use the KPI table as the basis for business recommendations rather than reporting raw totals alone.